In [ ]:
SEED = 2137

import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt

from sklearn import linear_model
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

data = pd.read_csv("data/domy.csv", decimal='.')
data[data == '?'] = pd.NA
# print(data.info())
# data.describe()

In [ ]:
data = data.drop("Id", axis="columns")
data = data[data["GrLivArea"] < 4_000]

data = data.drop("Alley", axis="columns") # too many '?'

data = data.drop("Utilities", axis="columns") # little distinct values
data = data.drop("Condition2", axis="columns") # little distinct values
data = data.drop("RoofMatl", axis="columns") # little distinct values
data = data.drop("Fence", axis="columns") # little distinct values
data = data.drop("MasVnrType", axis="columns") # insignificant
data = data.drop("MasVnrArea", axis="columns") # insignificant

data = data.drop("BsmtFinType1", axis="columns") # insignificant, covered by BsmtUnfSF and TotalBsmtSF
data = data.drop("BsmtFinSF1", axis="columns") # insignificant,  covered by BsmtUnfSF and TotalBsmtSF
data = data.drop("BsmtFinType2", axis="columns") # insignificant,  covered by BsmtUnfSF and TotalBsmtSF
data = data.drop("BsmtFinSF2", axis="columns") # insignificant,  covered by BsmtUnfSF and TotalBsmtSF

data = data.drop("Electrical", axis="columns") # insignificant, dependent on year built

In [ ]:
def fill_missing(column, method='interp'):
    if method == 'fill':
        data[column] = data[column].ffill()
        data[column] = data[column].bfill()
    else:
        data[column] = data[column].interpolate()

def qual_to_val(column, mapping):
    data[column] = data[column].map(mapping)
    data[column] = pd.to_numeric(data[column])
    fill_missing(column)

In [ ]:
qual_to_val("LotShape", {"Reg": 4, "IR1": 3, "IR2": 2, "IR3": 1})
qual_to_val("CentralAir", {"Y": 1, "N": 0})
qual_to_val("GarageFinish", {"Fin": 3, "RFn": 2, "Unf": 1, pd.NA: 0})
qual_to_val("PavedDrive", {"Y": 2, "P": 1, "N": 0})

In [ ]:
for col in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]:
    qual_to_val(col, {"Ex": "6", "Gd": "5", "TA": "4", "Fa": "3", "Po": "2", "NA": "0", pd.NA: "0"})

def question_marks(column):
    data[column] = pd.to_numeric(data[column])
    fill_missing(column)

for col in ["LotFrontage", "GarageYrBlt"]:
    question_marks(col)

data["MSSubClass"] = data["MSSubClass"].astype(str)

data["PricePerLivArea"] = data["SalePrice"] / data["GrLivArea"]

data = pd.get_dummies(data, dtype=int)

In [ ]:
# print(data.info(verbose=True))

sb.heatmap(data.iloc[:, :47].corr())

In [ ]:
y = data["SalePrice"]
y = data["PricePerLivArea"]
X = data.drop("SalePrice", axis="columns")
X = X.drop("PricePerLivArea", axis="columns")

In [ ]:
regressors = {
    "Linear": linear_model.LinearRegression(),
    "Ridge": linear_model.Ridge(random_state=SEED),
    "HistGB": HistGradientBoostingRegressor(random_state=SEED),
    "RandomForest": RandomForestRegressor(random_state=SEED),
}

for name, reg in regressors.items():
    result = cross_validate(reg, X, y)
    score = result["test_score"]
    avg = sum(score) / 5
    std = np.std(score)
    print(f"{name}: {score}, mean: {avg:.4}, stdev: {std:.4}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [ ]:
reg = linear_model.LinearRegression()
_ = reg.fit(X_train, y_train)
print(reg.score(X_test, y_test))
y_pred_linear = reg.predict(X_test)

In [ ]:
reg = linear_model.Ridge(random_state=SEED)
_ = reg.fit(X_train, y_train)
print(reg.score(X_test, y_test))
y_pred_ridge = reg.predict(X_test)

In [ ]:
reg = HistGradientBoostingRegressor(random_state=SEED)
_ = reg.fit(X_train, y_train)
print(reg.score(X_test, y_test))
y_pred_hist = reg.predict(X_test)

In [ ]:
reg = RandomForestRegressor(random_state=SEED)
_ = reg.fit(X_train, y_train)
print(reg.score(X_test, y_test))
y_pred_forest = reg.predict(X_test)

In [ ]:
plt.scatter(y_test, y_pred_linear, color="red", s=5)
plt.scatter(y_test, y_pred_ridge, color="green", s=5)
plt.scatter(y_test, y_pred_hist, color="purple", s=5)
plt.scatter(y_test, y_pred_forest, color="orange", s=5)

plt.plot([0, 250], [0, 250], color="blue")
plt.show()